# Ejemplos de volúmenes de datos

En este notebook aprenderemos cómo trabajar con volúmenes de datos en Databricks, utilizando SQL y Python.

### 1. Creación de un volumen desde SQL

Aunque se puede crear un volumen desde SQL, también se puede crear desde la interfaz de usuario de Databricks. Lo más importante es saber que se comporta como una carpeta normal

In [0]:
%sql
CREATE VOLUME IF NOT EXISTS workspace.liga.datos;

Una vez creado el volumen, podemos crear un fichero desde Python (un CSV por ejemplo) y guardarlo en el volumen. A continuación, podemos leerlo desde SQL y realizar consultas sobre él.

In [0]:
RUTA = "/Volumes/workspace/liga/datos"

# Escribir un fichero con Python estándar
with open(f"{RUTA}/jugadores.csv", "w") as f:
    f.write("nombre,equipo,goles\n")
    f.write("Lewandowski,Barcelona,12\n")
    f.write("Mbappé,Real Madrid,15\n")
    f.write("Griezmann,Atlético,9\n")

# Leerlo con pandas, sin nada específico de Databricks
import pandas as pd
jugadores = pd.read_csv(f"{RUTA}/jugadores.csv")
print(jugadores.sort_values("goles", ascending=False))

A continuación, podemos explorar el volumen desde Python

In [0]:
import os
print(os.listdir(RUTA))          # Python estándar

display(dbutils.fs.ls(RUTA))     # Utilidad de Databricks, con tamaños y fechas

Por último, podemos crear una tabla en SQL (Data Warehouse) desde un fichero (Data Lake)

In [0]:
%sql 
CREATE OR REPLACE TABLE workspace.liga.jugadores AS
SELECT * FROM read_files(
  '/Volumes/workspace/liga/datos/jugadores.csv',
  format => 'csv',
  header => true
);

### 02. Consultas híbridas (Data LakeHouse) 

Una de las ventajas de Databricks es que permite realizar consultas híbridas, es decir, podemos consultar datos que se encuentran en un volumen (Data Lake) y combinarlos con datos que se encuentran en una tabla (Data Warehouse). Esto nos permite aprovechar lo mejor de ambos mundos y realizar análisis más completos y eficientes.

In [0]:
%sql
SELECT p.equipo_local, p.goles_local, j.nombre, j.goles
FROM workspace.liga.partidos p                      -- tabla Delta gobernada
JOIN read_files('/Volumes/workspace/liga/datos/jugadores.csv',
                format => 'csv', header => true) j  -- fichero crudo en un volumen
  ON p.equipo_local = j.equipo;